Let's start with roughly analysing the tables printing some charateristics.

In [57]:
import pandas as pd

prices1 = pd.read_csv('../data/prices1.csv')
prices2 = pd.read_csv('../data/prices2.csv')
weather = pd.read_csv('../data/weather.csv')

print('prices1 details')
print(f'Rows and Columns:\n{prices1.shape}\n')
print(f'Column Names:\n{prices1.columns.tolist()}\n')
print(f'Data types:\n{prices1.dtypes}\n')
print(f'Samples:\n{prices1.sample(5)}')
print('-----------------------------------------------------------------------------------------------\n')

print('prices2 details')
print(f'Rows and Columns:\n{prices2.shape}\n')
print(f'Column Names:\n{prices2.columns.tolist()}\n')
print(f'Data types:\n{prices2.dtypes}\n')
print(f'Samples:\n{prices2.sample(5)}')
print('-----------------------------------------------------------------------------------------------\n')

print('weather details')
print(f'Rows and Columns:\n{weather.shape}\n')
print(f'Column Names:\n{weather.columns.tolist()}\n')
print(f'Data types:\n{weather.dtypes}\n')
print(f'Samples:\n{weather.sample(5)}')
print('-----------------------------------------------------------------------------------------------')

prices1 details
Rows and Columns:
(16703, 11)

Column Names:
['delivery_date', 'id', 'business_date', 'time', 'time_interval', 'fixing_i_price', 'fixing_i_volume', 'fixing_ii_price', 'fixing_ii_volume', 'continous_trading_WAvg_price', 'continous_trading_volume']

Data types:
delivery_date                       str
id                                int64
business_date                       str
time                                str
time_interval                       str
fixing_i_price                  float64
fixing_i_volume                 float64
fixing_ii_price                 float64
fixing_ii_volume                float64
continous_trading_WAvg_price    float64
continous_trading_volume        float64
dtype: object

Samples:
                   delivery_date     id business_date      time time_interval  \
7069   2024-10-21 00:00:00+00:00  17744    2024-10-20  00:00:00           0-1   
14524  2025-08-28 00:00:00+00:00  21915    2025-08-27  19:00:00         19-20   
7511   2024-11-09

One quick thing we can do is to rename the weather columns in English to have consistency with the others.

In [58]:
weather.rename(columns={'czas': 'time', 'id': 'id', 'miasto': 'city', 'temperatura': 'temperature', 
                        'wilgotnosc': 'humidity', 'opady': 'precipitation', 'naslonecznienie': 'sunny', 
                        'predkosc_wiatru': 'wind_speed'}, inplace=True)
print(f'Renamed weather columns: {weather.columns.tolist()}')

Renamed weather columns: ['time', 'id', 'city', 'temperature', 'humidity', 'precipitation', 'sunny', 'wind_speed']


Now let's work a bit with the time fields.

For the price1 table we have the following fields:
    delivery_date                       str (interpretation-> when electricity is delivered)
    business_date                       str (interpretation-> when the market auction happened)
    time                                str (interpretation-> when electricity is acutioned/delivered)
    time_interval                       str (interpretation-> the time interval in which the electricity has been auctioned)

The assumption is that people trade electricity on day x (business_date) and then the same electricity is delivered on day x+1 (delivery_date). For the time field I assume that electricity price is assigned in that hour and than delivered in the same hour.

For the price2 table we have the following fields:
    delivery_date                       str
    business_date                       str
    time                                str
    time_interval                       str

And for weather we have only: time

In [59]:
print(prices1[["delivery_date", "business_date", "time", "time_interval"]].head(20))
print(prices1["time"].unique())
print(prices1["time_interval"].unique())

print(prices2[["delivery_date", "business_date", "time", "time_interval"]].head(20))
print(prices2["time"].unique())
print(prices2["time_interval"].unique())

                delivery_date business_date      time time_interval
0   2024-01-01 00:00:00+00:00    2023-12-31  19:00:00         19-20
1   2024-01-01 00:00:00+00:00    2023-12-31  18:00:00         18-19
2   2024-01-01 00:00:00+00:00    2023-12-31  23:00:00         23-24
3   2024-01-01 00:00:00+00:00    2023-12-31  17:00:00         17-18
4   2024-01-01 00:00:00+00:00    2023-12-31  16:00:00         16-17
5   2024-01-01 00:00:00+00:00    2023-12-31  15:00:00         15-16
6   2024-01-01 00:00:00+00:00    2023-12-31  14:00:00         14-15
7   2024-01-01 00:00:00+00:00    2023-12-31  13:00:00         13-14
8   2024-01-01 00:00:00+00:00    2023-12-31  12:00:00         12-13
9   2024-01-01 00:00:00+00:00    2023-12-31  11:00:00         11-12
10  2024-01-01 00:00:00+00:00    2023-12-31  10:00:00         10-11
11  2024-01-01 00:00:00+00:00    2023-12-31  09:00:00          9-10
12  2024-01-01 00:00:00+00:00    2023-12-31  20:00:00         20-21
13  2024-01-01 00:00:00+00:00    2023-12-31  07:

We can notice that time_interval is just a label for time so we can think to combine the time with delivery date to merge those informations into one.

In [100]:
# With dt.normalize() we set the time to 00:00:00, then we add the right time component using pd.to_timedelta()
prices1["delivery_timestamp"] = pd.to_datetime(prices1["delivery_date"].astype(str)).dt.normalize() + pd.to_timedelta(prices1["time"].astype(str))
prices2["delivery_timestamp"] = pd.to_datetime(prices2["delivery_date"].astype(str)).dt.normalize() + pd.to_timedelta(prices2["time"].astype(str))
weather["time"] = pd.to_datetime(weather["time"])

print(prices1[["delivery_date", "business_date", "time", "time_interval", "delivery_timestamp"]].head())
print(prices2[["delivery_date", "business_date", "time", "time_interval", "delivery_timestamp"]].head())
print(weather[["time"]].head())


               delivery_date business_date      time time_interval  \
0  2024-01-01 00:00:00+00:00    2023-12-31  19:00:00         19-20   
1  2024-01-01 00:00:00+00:00    2023-12-31  18:00:00         18-19   
2  2024-01-01 00:00:00+00:00    2023-12-31  23:00:00         23-24   
3  2024-01-01 00:00:00+00:00    2023-12-31  17:00:00         17-18   
4  2024-01-01 00:00:00+00:00    2023-12-31  16:00:00         16-17   

         delivery_timestamp  
0 2024-01-01 19:00:00+00:00  
1 2024-01-01 18:00:00+00:00  
2 2024-01-01 23:00:00+00:00  
3 2024-01-01 17:00:00+00:00  
4 2024-01-01 16:00:00+00:00  
               delivery_date business_date      time    time_interval  \
0  2025-10-01 00:00:00+00:00    2025-09-30  20:15:00  01-10-25_Q20:15   
1  2025-10-01 00:00:00+00:00    2025-09-30  16:15:00  01-10-25_Q16:15   
2  2025-10-01 00:00:00+00:00    2025-09-30  16:00:00  01-10-25_Q16:00   
3  2025-10-01 00:00:00+00:00    2025-09-30  15:45:00  01-10-25_Q15:45   
4  2025-10-01 00:00:00+00:00    20

Let's look fro some duplicate presence:

In [110]:
print(f'Duplicates timestamps in prices1: {prices1["delivery_timestamp"].duplicated().sum()}')
# Here we show an example of duplicated timestamp in prices1.
print(prices1[prices1["delivery_timestamp"].duplicated(keep=False) == True].sort_values(["delivery_timestamp"]).head(10))
print('-----------------------------------------------------------------------------------------------\n')

print(f'Duplicates timestamps in prices2: {prices2["delivery_timestamp"].duplicated().sum()}')
# Here we show an example of duplicated timestamp in prices2.
print(prices2[prices2["delivery_timestamp"].duplicated(keep=False) == True].sort_values(["delivery_timestamp"]).head(10))
print('-----------------------------------------------------------------------------------------------\n')

print(f'Duplicates timestamps in weather: {weather.duplicated(["time", "city"]).sum()}')


Duplicates timestamps in prices1: 24
                   delivery_date     id business_date      time time_interval  \
16606  2025-11-24 00:00:00+00:00  24008    2025-11-23  00:00:00         00-01   
16628  2025-11-24 00:00:00+00:00  23960    2025-11-23  00:00:00         00-01   
16627  2025-11-24 00:00:00+00:00  23961    2025-11-23  01:00:00         01-02   
16604  2025-11-24 00:00:00+00:00  24009    2025-11-23  01:00:00         01-02   
16587  2025-11-24 00:00:00+00:00  24010    2025-11-23  02:00:00         02-03   
16626  2025-11-24 00:00:00+00:00  23962    2025-11-23  02:00:00         02-03   
16590  2025-11-24 00:00:00+00:00  24011    2025-11-23  03:00:00         03-04   
16625  2025-11-24 00:00:00+00:00  23963    2025-11-23  03:00:00         03-04   
16589  2025-11-24 00:00:00+00:00  24012    2025-11-23  04:00:00         04-05   
16624  2025-11-24 00:00:00+00:00  23964    2025-11-23  04:00:00         04-05   

       fixing_i_price  fixing_i_volume  fixing_ii_price  fixing_ii_volu